# 01. Audit external imagery

Read completed development and external-test collections, screen primary JECAM and all completed supplemental test fields for overlap, and audit imagery using the existing rules. No downloads, cloud queries, imports, field selection or training occur. Finish pending batches before running; collection changes invalidate the snapshot. Each audit creates a new analysis run, preserving earlier results.


**Input:** Completed collection or imported batches. **Output:** `quality.json`. **Next:** Preprocessing 02.

[Run order and reruns](../../docs/workflow.md).


In [1]:
from pathlib import Path
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import preprocessing_workflow as workflow
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows import seasons


## Run the local audit

The existing campaign folders remain authoritative. The collection 08 report summarizes them; no manual copying into the output layout is needed. JECAM bundles from teammates must already be imported before starting this notebook.


In [2]:
quality_handoff = workflow.audit(root)
print("Quality handoff:", quality_handoff)


Regions audited: 7321
Composite outcomes: {'quality_pass': 134209, 'low_coverage': 41495}
Composite timeline rows: 175704
Excluded 0 overlapping JECAM parcels. Report: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/jecam_overlap.json
Quality handoff: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/quality.json


## Coverage and next step

Counts here describe input fields and tile intervals. Cloud/shadow tables below measure pixel-date losses before compositing; legacy diagnostics remain unavailable where no pre-mask counts were saved. Preprocessing 02–03 determine complete windows and final capped samples.


In [3]:
show_table(seasons.read_table(quality_handoff.parent / "quality_counts.csv"), title="Imagery quality by crop and partition")
print("Source profile:", quality_handoff.parent / "reference_profile.csv")
print("Test overlap exclusions:", quality_handoff.parent / "jecam_overlap.json")
print("Organized outputs:", root / "pipeline_outputs/preprocessing" / quality_handoff.parent.name / "quality")
print("Next: 02_window_selection.ipynb uses this verified handoff automatically.")


Partition,Crop,Input parcel groups,Selected pixels across fragments,Audited tile intervals,Quality pass tile intervals,Other tile intervals
Training,Carrot,1401,541239,48960,36164,12796
Training,Cabbage,1040,186464,31632,23349,8283
Training,Potato,764,216602,23808,19929,3879
Validation,Carrot,453,159755,15840,11797,4043
Validation,Cabbage,324,56368,10200,7434,2766
Validation,Potato,290,95835,8952,7619,1333
Reserved test,Carrot,43,118,1032,829,203
Reserved test,Cabbage,56,1172,1368,1250,118
Reserved test,Potato,56,2585,1416,1183,233


Source profile: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/reference_profile.csv
Test overlap exclusions: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/analysis/external/external_jqin0imq/jecam_overlap.json
Organized outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/preprocessing/external_jqin0imq/quality
Next: 02_window_selection.ipynb uses this verified handoff automatically.


## Evidence and interpretation

Coverage describes this saved snapshot. Green-up is estimated timing; sample targets do not establish accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.


## Figures

Reference parcels by source and crop, and the share of half-month composites passing the 80% valid-pixel rule. PNGs are saved in the run's `figures/` folder.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

figure_folder = quality_handoff.parent / "figures"
profile_path = quality_handoff.parent / "reference_profile.csv"
if profile_path.is_file():
    show_and_save(figures.draw_reference_profile(figures.read_table(profile_path)), figure_folder, "reference_parcels")
else:
    print("Reference profile not saved for this run; figure skipped:", profile_path)
show_and_save(figures.draw_quality_pass_rates(figures.read_table(quality_handoff.parent / "quality_counts.csv")),
              figure_folder, "quality_pass_rates")

## Cloud and shadow loss before compositing

A **pixel-date** is one selected parcel pixel on one Sentinel-2 acquisition date. Each pixel-date is either kept as clear, removed as cloud or shadow (SCL classes and cloud probability), or removed for another reason. The first table groups pixel-dates by partition and crop; the second groups them by calendar half-month, all source years together. Tile intervals from archives without pre-mask counts are counted in `intervals_without_pre_mask_counts` and reported as unavailable, never as zero loss.

In [ ]:
# Show cloud/shadow loss tables; runs audited before these tables existed are computed from quality.json instead
loss_path = quality_handoff.parent / "observation_loss.csv"
loss_month_path = quality_handoff.parent / "observation_loss_by_half_month.csv"
if loss_path.is_file() and loss_month_path.is_file():
    loss_by_crop = seasons.read_table(loss_path)
    loss_by_half_month = seasons.read_table(loss_month_path)
else:
    print("Loss tables not saved for this run; computing them from quality.json (nothing is written to the run).")
    loss_by_crop, loss_by_half_month, _ = workflow.observation_loss(json.loads(quality_handoff.read_text()))
show_table(loss_by_crop, title="Pixel-date losses by partition and crop")
show_table(loss_by_half_month, title="Pixel-date losses by calendar half-month")
show_and_save(figures.draw_observation_loss(loss_by_half_month), figure_folder, "cloud_shadow_by_half_month")

## Evidence: reference parcels on downloaded imagery

One parcel per source and crop (median selected-pixel count, chosen without looking at model results). The red line is the reference parcel outline from the source dataset. The coloured cells are the Sentinel-2 pixels actually downloaded into the annual TIFF, shown as NDVI for the eight Jan–Apr half-months. Empty cells failed the cloud mask or were not selected. Loading `quality.json` can take a minute for large runs.

In [ ]:
show_parcel_examples = True  # Set False to skip loading the full quality handoff

if show_parcel_examples:
    quality_data = json.loads(quality_handoff.read_text())
    parcel_examples = figures.parcel_evidence_examples(quality_data)
    del quality_data
    for example in parcel_examples:
        show_and_save(figures.draw_parcel_evidence(example), figure_folder,
                      f"parcel_{example['source']}_{example['crop']}")
    if not parcel_examples:
        print("No parcels with saved annual TIFFs in this handoff.")